# BindingDB 蛋白序列相似性约束切分分析

本 notebook 展示 BindingDB sequence-based 数据集从原始表到 `train / valid / test` 的切分过程，并验证不同 split 之间是否存在 `identity >= 40%` 且覆盖度满足阈值的蛋白序列相似性命中。

核心原则：流式读取只负责处理 8GB 级别原始 TSV；真正避免数据泄露的是 **protein sequence clustering + component-level split + independent validation**。

## 1. 环境和路径设置

下面的参数默认读取一个小规模 demo 输出目录。完整实验时，将 `OUTPUT_DIR` 改为 `seq/processed/bindingdb_sequence_split`，并在命令行或 notebook 中运行完整管线。

In [ ]:
from pathlib import Path
import json
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 从 notebook 所在的 seq/ 目录回到项目根目录。
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "seq":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_TSV = PROJECT_ROOT / "seq/data/BindingDB_All.tsv"
SCRIPT = PROJECT_ROOT / "seq/scripts/data/bindingdb_sequence_split.py"

# demo 目录来自前 5000 行原始 TSV，适合快速检查 notebook 是否能跑通。
# 完整实验建议改成 PROJECT_ROOT / "seq/processed/bindingdb_sequence_split"。
OUTPUT_DIR = PROJECT_ROOT / "seq/processed/bindingdb_sequence_split_demo"

MIN_SEQ_ID = 0.4
COVERAGE = 0.8

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_TSV exists:", RAW_TSV.exists())
print("SCRIPT exists:", SCRIPT.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)

## 2. 可选：运行小规模 demo 管线

这一步会读取前 5000 行 BindingDB 原始 TSV，并完整执行 `clean -> cluster -> split -> validate`。完整 8.2GB 数据不建议直接在 notebook 中运行，推荐在终端执行 README 中的 `run-all` 命令。

In [ ]:
RUN_DEMO_PIPELINE = True

if RUN_DEMO_PIPELINE:
    cmd = [
        "python", str(SCRIPT), "run-all",
        "--raw-tsv", str(RAW_TSV),
        "--output-dir", str(OUTPUT_DIR),
        "--affinity-types", "Ki",
        "--limit-rows", "5000",
        "--min-seq-id", str(MIN_SEQ_ID),
        "--coverage", str(COVERAGE),
        "--train-ratio", "0.8",
        "--valid-ratio", "0.1",
        "--test-ratio", "0.1",
        "--seed", "2026",
    ]
    # check=True 让错误直接暴露，避免后续读取旧结果。
    subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
else:
    print("跳过运行管线，直接读取已有输出。若输出不存在，请将 RUN_DEMO_PIPELINE 改为 True 或先在终端运行脚本。")

## 3. 读取报告和切分产物

四个 JSON 报告分别对应：清洗、蛋白聚类、component 切分、cross-split 相似性验证。`bindingdb_clean_with_split.csv` 是后续训练可以直接使用的总表。

In [ ]:
def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

clean_report = load_json(OUTPUT_DIR / "clean_report.json")
cluster_report = load_json(OUTPUT_DIR / "cluster_report.json")
split_report = load_json(OUTPUT_DIR / "split_report.json")
validation_report = load_json(OUTPUT_DIR / "validation_report.json")

clean_df = pd.read_csv(OUTPUT_DIR / "bindingdb_clean.csv")
split_df = pd.read_csv(OUTPUT_DIR / "bindingdb_clean_with_split.csv")
components_df = pd.read_csv(OUTPUT_DIR / "protein_components.csv")
component_splits_df = pd.read_csv(OUTPUT_DIR / "component_splits.csv")
leakage_hits_df = pd.read_csv(OUTPUT_DIR / "cross_split_leakage_hits.csv")

print("clean rows:", len(clean_df))
print("split rows:", len(split_df))
print("unique proteins:", split_df["protein_id"].nunique())
print("components:", components_df["component_id"].nunique())
print("leakage report:", validation_report)

输出解释：

- `clean rows` 是过滤和重复聚合后的 BindingDB 样本数。
- `unique proteins` 是参与切分的唯一蛋白序列数。
- `components` 是在 40% identity 阈值下形成的蛋白相似性连通分量数量。
- `leakage report` 中 `hit_count=0` 且 `leakage_detected=False` 才表示 cross-split 相似性验证通过。

## 4. 清洗阶段漏斗图

这里展示从原始行数到最终 clean 样本数的减少过程。被过滤的主要原因通常包括：多链 target、缺少指定亲和力类型、亲和力不是明确数字。

In [ ]:
funnel = pd.Series({
    "raw_rows_seen": clean_report["raw_rows_seen"],
    "kept_before_aggregation": clean_report["kept_rows_before_aggregation"],
    "aggregated_rows": clean_report["aggregated_rows"],
    "unique_proteins": clean_report["unique_proteins"],
})

ax = funnel.plot(kind="bar", figsize=(8, 4), color=["#4C78A8", "#72B7B2", "#54A24B", "#E45756"])
ax.set_title("BindingDB cleaning funnel")
ax.set_ylabel("count")
ax.bar_label(ax.containers[0], fmt="%.0f")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

skip_cols = [c for c in clean_report if c.startswith("skipped_")]
pd.Series({c: clean_report[c] for c in skip_cols}).sort_values(ascending=False)

输出解释：

漏斗图告诉我们有多少原始 BindingDB 行真正进入训练候选集。下面的 `skipped_*` 表格用于定位过滤损失来自哪里；如果样本太少，可以考虑放宽单链限制、改用 `Kd` 或单独构建 `IC50` 数据集。

## 5. 标签和蛋白长度分布

这一步检查回归标签 `p_affinity_median` 和蛋白序列长度。异常长蛋白会显著增加 ESM-2 embedding 成本，后续可考虑截断、滑窗或单独过滤。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

split_df["p_affinity_median"].hist(ax=axes[0], bins=30, color="#4C78A8")
axes[0].set_title("pAffinity distribution")
axes[0].set_xlabel("pAffinity")
axes[0].set_ylabel("sample count")

split_df["protein_length"].hist(ax=axes[1], bins=30, color="#F58518")
axes[1].set_title("protein length distribution")
axes[1].set_xlabel("amino-acid length")
axes[1].set_ylabel("sample count")

plt.tight_layout()
plt.show()

split_df[["p_affinity_median", "protein_length", "replicate_count"]].describe()

输出解释：

- `p_affinity_median` 越大表示结合越强。
- `protein_length` 分布用于估计 PLM embedding 的显存和时间成本。
- `replicate_count` 表示同一蛋白-配体-标签类型聚合了多少条实验记录。

## 6. 蛋白相似性 component 分布

MMseqs 在 `identity >= 40%`、`coverage >= 80%` 条件下找到相似蛋白边。脚本把这些边连接成 component，并保证一个 component 只进入一个 split。

In [ ]:
component_sizes = components_df.drop_duplicates("component_id")["component_size"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
component_sizes.hist(ax=axes[0], bins=30, color="#54A24B")
axes[0].set_title("protein component size distribution")
axes[0].set_xlabel("proteins per component")
axes[0].set_ylabel("component count")

top_components = component_splits_df.sort_values("sample_count", ascending=False).head(15)
axes[1].barh(top_components["component_id"], top_components["sample_count"], color="#B279A2")
axes[1].invert_yaxis()
axes[1].set_title("top components by sample count")
axes[1].set_xlabel("sample count")

plt.tight_layout()
plt.show()

cluster_report

输出解释：

如果最大 component 很大，那么 train/valid/test 的样本比例可能无法完全接近 8/1/1，因为这个大 component 不能拆开。这个约束是为了优先保证蛋白相似性不泄露。

## 7. 蛋白聚类散点图

下面用散点图展示蛋白聚类后的 component 结构。左图中每个点代表一个唯一蛋白，横轴是 component 排名，纵轴是蛋白长度，颜色表示该 component 被分到的 split；右图中每个点代表一个 component，横轴是 component 内蛋白数，纵轴是该 component 覆盖的样本数。

这个图的目的不是证明具体序列相似性数值，而是直观看到：相似蛋白被聚成 component，并且一个 component 只进入一个 split。真正的泄露判定仍以后面的 cross-split MMseqs 验证为准。

In [ ]:
component_rank = (
    component_splits_df.sort_values(["sample_count", "protein_count", "component_id"], ascending=[False, False, True])
    .reset_index(drop=True)
    .assign(component_rank=lambda df: np.arange(len(df)))
    [["component_id", "component_rank", "split", "sample_count", "protein_count"]]
)

protein_meta = split_df.drop_duplicates("protein_id")[["protein_id", "protein_length", "target_name", "uniprot_id"]]
protein_cluster_plot = (
    components_df.merge(component_rank, on="component_id", how="left")
    .merge(protein_meta, on="protein_id", how="left")
)

# 全量数据时唯一蛋白可能很多，绘图时抽样以保证 notebook 响应速度。
max_points = 5000
if len(protein_cluster_plot) > max_points:
    plot_proteins = protein_cluster_plot.sample(max_points, random_state=2026).copy()
else:
    plot_proteins = protein_cluster_plot.copy()

rng = np.random.default_rng(2026)
plot_proteins["component_rank_jitter"] = plot_proteins["component_rank"] + rng.normal(0, 0.08, size=len(plot_proteins))

split_colors = {"train": "#4C78A8", "valid": "#F58518", "test": "#54A24B"}
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for split_name, group in plot_proteins.groupby("split"):
    axes[0].scatter(
        group["component_rank_jitter"],
        group["protein_length"],
        s=np.clip(group["component_size"].astype(float), 8, 80),
        alpha=0.65,
        label=split_name,
        color=split_colors.get(split_name, "#999999"),
        edgecolors="none",
    )
axes[0].set_title("Proteins grouped by sequence-similarity component")
axes[0].set_xlabel("component rank by sample count")
axes[0].set_ylabel("protein length")
axes[0].legend(title="split")

for split_name, group in component_rank.groupby("split"):
    axes[1].scatter(
        group["protein_count"],
        group["sample_count"],
        s=np.clip(np.sqrt(group["sample_count"].astype(float)) * 10, 20, 250),
        alpha=0.75,
        label=split_name,
        color=split_colors.get(split_name, "#999999"),
        edgecolors="white",
        linewidths=0.5,
    )
axes[1].set_title("Components by protein count and sample count")
axes[1].set_xlabel("proteins per component")
axes[1].set_ylabel("samples per component")
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].legend(title="split")

plt.tight_layout()
plt.show()

component_rank.head(10)

输出解释：

- 左图中，同一竖向区域代表同一个相似性 component；颜色只有一种，说明该 component 整体进入同一个 split。
- 右图中，越靠右说明 component 内蛋白越多，越靠上说明该 component 覆盖样本越多。
- 如果出现非常大的 component，split 比例可能偏离 8:1:1；这是为了避免拆开相似蛋白造成泄露。

## 8. Train / Valid / Test 分布

这里同时展示样本数、唯一蛋白数和 component 数。注意 split 是在 component 层面完成的，不是随机按样本行切分。

In [ ]:
split_summary = pd.DataFrame({
    "samples": split_df.groupby("split").size(),
    "proteins": split_df.groupby("split")["protein_id"].nunique(),
    "components": component_splits_df.groupby("split")["component_id"].nunique(),
}).reindex(["train", "valid", "test"]).fillna(0).astype(int)

ax = split_summary.plot(kind="bar", figsize=(8, 4), color=["#4C78A8", "#F58518", "#54A24B"])
ax.set_title("split distribution")
ax.set_ylabel("count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

split_summary

输出解释：

如果 `valid` 或 `test` 样本数偏离目标比例，通常是因为大 component 不能拆分。这种偏离比蛋白泄露更可接受，因为实验目标是严格评估泛化。

## 9. Cross-split 相似性验证

这一步读取独立验证结果。验证不是简单相信 component split，而是重新对 train/valid/test 之间的蛋白 FASTA 做 MMseqs search。

In [ ]:
print(json.dumps(validation_report, indent=2, ensure_ascii=False))

if validation_report["leakage_detected"]:
    display(leakage_hits_df.head(20))
else:
    print("验证通过：未发现 cross-split protein sequence identity >= 40% 且 coverage >= 80% 的命中。")

pair_counts = leakage_hits_df.groupby(["query_split", "target_split"]).size().unstack(fill_value=0) if not leakage_hits_df.empty else pd.DataFrame()
pair_counts

输出解释：

最终可以在报告中写成：在 `min_seq_id = 0.4`、`coverage = 0.8` 条件下，独立 cross-split MMseqs 检查得到 `hit_count = 0`，因此该切分没有检测到训练、验证、测试之间的蛋白序列相似性泄露。若 `hit_count > 0`，则必须重新生成 component 或调整参数，不能声称满足 `<40%`。

## 10. 后续训练文件

模型训练阶段应读取以下三个文件：

```text
seq/processed/bindingdb_sequence_split/train.csv
seq/processed/bindingdb_sequence_split/valid.csv
seq/processed/bindingdb_sequence_split/test.csv
```

每行包含 `ligand_smiles`、`protein_sequence` 和 `p_affinity_median`。第一版 baseline 可以直接使用：

```text
protein_sequence -> ESM-2 frozen embedding
ligand_smiles -> Morgan fingerprint
p_affinity_median -> regression label
```